Makes a summarized `.tsv` file based on chosen atlas and map.

In [35]:
# Import packages
from nilearn import plotting as npl
import SUITPy as suit
import SUITPy.atlas as atlas
import nibabel as nib
import ants
import matplotlib.pyplot as plt
import numpy as np

from pathlib import Path # for saving file name

import pandas as pd

In [ ]:
# tissues: 'gm', 'wm', 'csf', 
# "tissue" is a bad name...think of something more accurate to incl csf, etc.
tissue = 'gm' # default
tissue_dict = {
    'gm': 'c1', # grey matter
    'wm': 'c2', # white matter
    'csf': 'c3' # cerebral spinal fluid
}

# directories
anat_dir = 'smarts_cerebellum/anatomicals'
p_df = pd.read_csv('smarts_cerebellum/anatomicals/participants_anat.tsv', sep = '\t')

# should I store these results in a new folder or in the old folder?
# maybe inside the anats directory, inside a folder called f'{tissue}_results' so that new folder for each tissue

for i in range(0, p_df.shape[0]):

    p_id = p_df['ID'].iloc[i]
    week = (p_df['Week'].iloc[i]).strip() # sometimes have extra white spaces
    p_centre = (str(p_df['Centre'].iloc[i])).strip()
    refT1 = (p_df['RefT1'].iloc[i]).strip()

    subj_id = f'{p_centre.strip()}_{p_id}'

    # path to store results from this pipeline for each participant + timepoint
    results_path = Path(anat_dir)/subj_id/week

    t1_path = f'{anat_dir}/{subj_id}/{week}/{subj_id}_{week}_T1.nii'
    tissue_path = f'{anat_dir}/{subj_id}/{week}/{tissue_dict[tissue]}{subj_id}_{week}_T1.nii'

    # check that paths exist
    if not Path(t1_path).is_file():
        print(f'T1 path does not exist for {subj_id} in week {week}')
        continue
    if not Path(tissue_path).is_file():
        print(f'{tissue} path does not exist for {subj_id} in week {week}')
        continue

    tissue_vol = f'{results_path}/{subj_id}_{week}_T1_{tissue}_vol.nii'
    tissue_vol_img = nib.load(tissue_vol)
    
    # note that the volume of voxels in this image and the output from reslice are the same, since they're both in SUIT space
    voxel_dim = tissue_vol_img.header.get_zooms()[:3]
    voxel_vol = np.prod(voxel_dim)

    atlas.fetch_atlas('Nettekoven_2024')
    df = atlas.summarize_data(tissue_vol_img,
                              space = 'SUIT',
                              stats = ['nanmean'],
                              atlas = 'Nettekoven_2024',
                              maps = 'atl-NettekovenAsym32'
                              )
    df[f'{tissue}v'] = df['nanmean'] * (df['size']/voxel_vol)

    df.rename(columns={'nanmean': f'avg_{tissue}v'}, inplace = True)
    df['image_name'] = f'{tissue}v_img_{subj_id}_{week}_T1.nii'
    df['subj_id'] = subj_id
    df['week'] = week
    df.head(5)

    # put all of this in a new dataframe
    if i == 0:
        # header for only the first run
        all_df = df
    else:
        all_df = pd.concat([all_df, df], ignore_index = True)

    # select the row to write descriptive data to
    row_mask = (all_df['subj_id']==subj_id) & (all_df['week']==week)
    print(f'Writing data for {subj_id} at {week}')

    all_df.loc[row_mask, 'ID'] = p_id
    all_df.loc[row_mask, 'Week'] = week
    all_df.loc[row_mask, 'Centre'] = p_centre
    all_df.loc[row_mask, 'RefT1'] = refT1
    all_df.loc[row_mask, 'age'] = str(p_df['age'].iloc[i])
    all_df.loc[row_mask, 'Gender'] = p_df['Gender'].iloc[i]
    all_df.loc[row_mask, 'isPatient'] = str(p_df['isPatient'].iloc[i])
    all_df.loc[row_mask, 'LesionSide'] = p_df['LesionSide'].iloc[i]
    all_df.loc[row_mask, 'LesionLocation'] = p_df['LesionLocation'].iloc[i]
    all_df.loc[row_mask, 'handedness'] = str(p_df['handedness'].iloc[i])

all_df.to_csv(f'{anat_dir}/{tissue}v_atlas_summarized.tsv', mode = 'w', sep = '\t', index = False, header = True)
